In [ ]:
from google.colab import drive
drive.mount(
    '/content/drive'
)

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub sentence-transformers faiss-cpu

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
import torch
print(torch.cuda.is_available())

# **Load Model**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16   # or "auto"
)

tokenizer.pad_token = tokenizer.eos_token

# **Generator configuration pipeline**

In [ ]:
# generator = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     return_full_text=False,
#     max_new_tokens=500,
#     do_sample=False,
#     pad_token_id=tokenizer.eos_token_id,
#     eos_token_id=tokenizer.eos_token_id
# )

# **Query**

In [ ]:
# import torch

# messages = [
#     {"role": "user", "content": "Tell me about Suman Saha's contribution to quantum oncology. Also, explicitly identify the claims of your response"}
# ]




In [ ]:
user_prompt = "Tell me about Suman Saha's contribution to quantum oncology. Also, explicitly identify the claims of your response"


In [ ]:
# prompt = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )

# output = generator(prompt)
# print(output[0]["generated_text"])

In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread
from IPython.display import display, Markdown
import torch

def stream_generate(user_prompt, max_new_tokens=500):
    messages = [{"role": "user", "content": user_prompt}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        timeout=None
    )

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    partial_text = ""
    handle = display(Markdown(""), display_id=True)

    for new_text in streamer:
        partial_text += new_text
        handle.update(Markdown(partial_text))

    thread.join()
    return partial_text

In [ ]:
response = stream_generate("Tell me about Suman Saha's contribution to quantum oncology. Also, explicitly identify the claims of your response.")

For Llama 3 model, use this without pipeline

In [ ]:
# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_tensors="pt",
#     return_dict=True
# ).to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=64,
#         do_sample=False,
#         use_cache=True,
#         pad_token_id=tokenizer.eos_token_id,
#         eos_token_id=tokenizer.eos_token_id
#     )

In [ ]:
# response = tokenizer.decode(
#     outputs[0][inputs["input_ids"].shape[-1]:],
#     skip_special_tokens=True
# )
# print(response)

# **External Documents**

In [ ]:
# text = """
# Interstellar is a 2014 epic science fiction film co-written,
# directed, and produced by Christopher Nolan.
# It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
# Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
# Set in a dystopian future where humanity is struggling to
# survive, the film follows a group of astronauts who travel
# through a wormhole near Saturn in search of a new home for
# mankind.
# Brothers Christopher and Jonathan Nolan wrote the screenplay,
# which had its origins in a script Jonathan developed in 2007.
# Caltech theoretical physicist and 2017 Nobel laureate in
# Physics[4] Kip Thorne was an executive producer, acted as a
# scientific consultant, and wrote a tie-in book, The Science of
# Interstellar.
# Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
# the Panavision anamorphic format and IMAX 70 mm.
# Principal photography began in late 2013 and took place in
# Alberta, Iceland, and Los Angeles.
# Interstellar uses extensive practical and miniature effects and
# the company Double Negative created additional digital effects.
# Interstellar premiered on October 26, 2014, in Los Angeles.
# In the United States, it was first released on film stock,
# expanding to venues using digital projectors.
# The film had a worldwide gross over $677 million (and $773
# million with subsequent re-releases), making it the tenth-highest
# grossing film of 2014.
# It received acclaim for its performances, direction, screenplay,
# musical score, visual effects, ambition, themes, and emotional
# weight.
# It has also received praise from many astronomers for its
# scientific accuracy and portrayal of theoretical astrophysics.
# Since its premiere, Interstellar gained a cult following,[5] and
# now is regarded by many sci-fi experts as one of the best
# science-fiction films of all time.
# Interstellar was nominated for five awards at the 87th Academy
# Awards, winning Best Visual Effects, and received numerous other
# accolades"""


In [ ]:
# texts = text.split(".")
# texts = [t.strip(' \n') for t in texts]

# **Load Embedding model**

In [ ]:
# # 1) Load local embedding model
# embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

In [ ]:
# doc_embeds = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
# print(doc_embeds.shape)

In [ ]:
# dim = doc_embeds.shape[1]
# index = faiss.IndexFlatIP(dim)   # use IP with normalized embeddings for cosine similarity
# index.add(np.float32(doc_embeds))

# **Chunks searching**

In [ ]:
# # 5) Search function
# def retrieve(query, top_k = 10):
#     query_embed = embedder.encode(
#         [query],
#         convert_to_numpy=True,
#         normalize_embeddings=True
#     )

#     scores, ids = index.search(np.float32(query_embed), top_k)

#     candidates = []


#     for rank, (doc_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
#       candidates.append({
#           "doc_id": int(doc_id),
#           "text": texts[int(doc_id)],
#           "retrieval_score": float(score),
#           "retrieval_rank": rank
#       })

#     return candidates

#     # texts_np = np.array(texts)
#     # results = pd.DataFrame({
#     #     "texts": texts_np[similar_item_ids[0]],
#     #     "score": scores[0]
#     # })

#     # print(f"Query: {query}\nNearest neighbors:")
#     # return results

# **Prepare the reranker**

In [ ]:
# reranker = CrossEncoder("BAAI/bge-reranker-base")

In [ ]:
# def rerank(query, candidates, final_k=5):
#     pairs = [[query, c["text"]] for c in candidates]
#     rerank_scores = reranker.predict(pairs)

#     for c, rs in zip(candidates, rerank_scores):
#         c["rerank_score"] = float(rs)

#     reranked = sorted(
#         candidates,
#         key=lambda x: x["rerank_score"],
#         reverse=True
#     )

#     return reranked[:final_k]

# **For larger corpora:**

retrieve_k=20 to 50
final_k=3 to 5

In [ ]:
# def search_with_rerank(query, retrieve_k=10, final_k = 3):
#   candidates = retrieve(query, top_k=retrieve_k)
#   top_docs = rerank(query, candidates, final_k=final_k)

#   return pd.DataFrame(top_docs)[[
#       "doc_id",
#       "text",
#       "retrieval_rank",
#       "retrieval_score",
#       "rerank_score"
#   ]]

In [ ]:
# query = "how precise was the science?"

# response = search_with_rerank(query, retrieve_k=10, final_k=3)

# response

In [ ]:
import nbformat

# Load your notebook file
file_path = '/content/drive/MyDrive/Colab Notebooks/LLM_02_RAG.ipynb'
with open(file_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Remove the problematic widgets metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']
    print("Metadata 'widgets' removed successfully.")

# Save the clean version
with open('clean_notebook.ipynb', 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)